# Week 1

## Downloaded and Manipulated Fits Files
We downloaded two fits files with 30 images each, then created a new difference fits file.

In [ ]:
import numpy as np

# Set up matplotlib
import matplotlib.pyplot as plt

from astropy.io import fits

# Extracting data from the light image file
light_image = fits.open(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 1\Linearity Images-20260518T043649Z-3-001\Linearity Images\lm_260429_000152.fits")
#light_image.info()
light_img_data = light_image[0].data
print(type(light_img_data)) # Show the Python type for image_data
print(light_img_data.shape) # Show the number of pixels per side in the 2-D image
light_image.close()

#Extracting data from the dark image file
dark_image = fits.open(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 1\Linearity Images-20260518T043649Z-3-001\Linearity Images\lm_260429_000162.fits")
dark_img_data = dark_image[0].data
dark_image.close()

#Plotting one frame from each set
plt.figure()
plt.imshow(light_img_data[1], cmap="gray")
plt.colorbar()

plt.figure()
plt.imshow(dark_img_data[1], cmap="gray")
plt.colorbar()

# plotting all frames from the light set together
fig1, axes1 = plt.subplots(6, 5, figsize = (18, 15))

axes1 = axes1.ravel()  # flatten the grid for easy indexing

for i in range(30):
    axes1[i].imshow(light_img_data[i], cmap="gray", origin="lower")
    axes1[i].set_title(f"Frame {i}", fontsize=10)
    axes1[i].axis("off")

print("fig1 is:", type(fig1))
fig1.suptitle("All Light Images", fontsize=30)
plt.tight_layout()
plt.show()

# plotting all frames from the dark set together
fig2, axes2 = plt.subplots(6, 5, figsize = (18, 15))
axes2 = axes2.ravel()  # flatten the grid for easy indexing

for i in range(30):
    axes2[i].imshow(dark_img_data[i], cmap="gray", origin="lower")
    axes2[i].set_title(f"Frame {i}", fontsize=10)
    axes2[i].axis("off")

fig2.suptitle("All Dark Images", fontsize=30)
plt.tight_layout()
plt.show()

# making a light - dark fits plot
net_light_data = light_img_data - dark_img_data

fig3, axes3 = plt.subplots(6, 5, figsize = (18, 15))
axes3 = axes3.ravel()  # flatten the grid for easy indexing

for i in range(30):
    axes3[i].imshow(net_light_data[i], cmap="gray", origin="lower")
    axes3[i].set_title(f"Frame {i}", fontsize=10)
    axes3[i].axis("off")

fig3.suptitle("Net Light Images", fontsize=30)
plt.tight_layout()
plt.show()

# making new net light FITS file
new_header = light_image[0].header.copy()
new_header['HISTORY'] = 'subtracted light image Im_260429_000152 from dark image Im_260429_000162'
# Write the FITS file
hdu = fits.PrimaryHDU(net_light_data, header=new_header)
hdu.writeto("net_light_images_week_1.fits", overwrite=True)

## Figures Created
![All light reads](Week_1/light_images_grid_wk1.png)  
![All dark reads](Week_1/dark_images_grid_wk1.png)  
![All difference reads](Week_1/net_images_grid_wk1.png)

### Notes
-this week I mostly worked to figure out the ds9 app and what fits files were  
-I wanted to see how the detector reads changed over time so I plotted all frames in one figure for the light, dark, and net images  
-I am still getting used to working with matplotlib so my code is quite messy here and I hope to improve my flow as I continue working  

# Week 2

## Downloaded Fits Files From LBTI Archive
I did this by selecting the files I wanted to download, then creating a .txt file of the urls to download in the interface then downloaded that to my computer. <br><br>
Next I went into my terminal and went into the folder I wanted my downloaded files to be in. Getting into the file in my terminal is done using basic bash:  
(in terminal) cd "path_to_desired_folder"<br><br>
Once in the folder I wanted to download to, I ran the following command in my terminal (this is for windows only but I'm sure a similar command can be looked up for mac, linux, etc.):  
(in terminal) foreach ($url in Get-Content "path_to_.txt_file") {
    Invoke-WebRequest -Uri $url -OutFile (Split-Path $url -Leaf)
}<br><br>
This method allowed me to download files in the background while working on other tasks on my computer.  
It did require my laptop to be on, so if I closed it or it went to sleep the downloading would stop mid .txt file and whatever .fits file it was downloading would be corrupted. This was solved by changing the settings of my laptop to not go to sleep while charging.<br><br>
If a .fits download was stopped halfway through, I just deleted that downloaded file, took the completed downloaded files out of the .txt file, and reran the above code.


## Averaged Multiple Fits Files and Made Average Difference Files For Different Modes
Using the log, I organized downloaded files based on mode (fast, medium, slow) and light vs dark files. I opened these files then found the mean of the light and dark images separately for each mode, then created an average difference file for each mode.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob


def create_file_list(file_paths):
    file_list = glob.glob(file_paths)
    return file_list


# getting an average file for a given list of files, carrying over header
# filelist is a list of the full fits files, filetype is light or dark files
def average_files(filelist, filetype):

    #making a list of the data from each fits file
    file_concat = [fits.getdata(image) for image in filelist]

    #finding the mean (averaging the files)
    #.astype(np.float32) shrinks the acuracy of the estimation, since the mean auto
    #defaults to float64, so this will make the file smaller
    final_file_data = np.mean(file_concat, axis=0).astype(np.float32)

    #carry over first image header and add information on averaging
    original_header = fits.getheader(filelist[0])
    new_header = original_header.copy()
    # will return something of the style: averaged 10 light files
    new_header['HISTORY'] = f"averaged {len(filelist)} {filetype} files"
    # gives numeber of averaged files
    new_header['NFILES'] = len(filelist)
    new_header['METHOD'] = 'MEAN'

    return final_file_data, new_header

# making a net light file, and adjusting the header to reflect this
def calculate_difference(lightdata, darkdata, header):
    # calculate difference in light and dark files
    net_data = lightdata - darkdata

    #adjust header for new net_data file
    new_header = header.copy()
    new_header['HISTORY'] = f"difference between averaged light and dark file data"
    return net_data, new_header

#creating and saving a new fits file to my computer
def create_file(filename, filedata, header):
    hdu = fits.PrimaryHDU(filedata, header=header)
    hdu.writeto(filename, overwrite=True)

if __name__ == "__main__":
    # I kept the basic structure and only changed the folder when creating 
    # the dark and light file list, with each folder assigned to fast, med, or slow
    light_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 2\Week 2 Fits\Slow Light\*.fits.gz")
    avg_light, light_header = average_files(light_files, "light")
    dark_files = create_file_list(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Downloaded Files\Week 2\Week 2 Fits\Slow Dark\*.fits.gz")
    avg_dark, dark_header = average_files(dark_files, "dark")
    net_data, net_header = calculate_difference(avg_light, avg_dark, light_header)
    create_file("avg_slow_light_image_week2.fits", avg_light, light_header)
    create_file("avg_slow_dark_image_week2.fits", avg_dark, dark_header)
    create_file("net_slow_image_week2.fits", net_data, net_header)

## Plotting The Median Pixel Value for a Box In Each Frame
In total an average dark, light, and difference file was created for each of the three modes.  
For each mode, I selected a box of pixels in each image to average using the median, then plotted the median pixel value in the box as a function of time as each frame moved forward in time. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# I chose to analyze a box from 970<x<1800 and 1070<y<1670 for all images
# In lab we discussed that this is a good choice since we want the box to 
# be as central as possible

# for a given fits file, cut down to the box, find the median of pixel values in this 
# box for each image, and add to the plot
# filename is the fits file, setting and color are both strings where setting describes
# the LBTI setting, file describes the fits file type, and color is the desired
# color of the plotted points for that file
def plot_avg(filename, setting, filetype, color):
    # make data list with only data from pixels in the defined box
    box_data = fits.getdata(filename)[:,970:1800,1070:1670]
    plt.title("Median Pixel Value Over Time: " + setting + " Mode", fontsize = 15)
    for i in range(box_data.shape[0]):
        # find the median of the pixels and plot for each image
        median_data = np.median(box_data[i,:,:])
        plt.scatter(i, median_data, c= color, label = filetype if i==0 else '')

if __name__ == "__main__":
    # fast plot
    plt.figure()
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_fast_light_image_week2.fits", "Fast", "light", "blue")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_fast_dark_image_week2.fits", "Fast", "dark", "red")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_fast_image_week2.fits", "Fast", "net", "black")
    plt.legend()
    plt.grid(True)
    plt.show()

    #medium plot
    plt.figure()
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_medium_light_image_week2.fits", "Medium", "light", "blue")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_medium_dark_image_week2.fits", "Medium", "dark", "red")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_medium_image_week2.fits", "Medium", "net", "black")
    plt.legend()
    plt.grid(True)
    plt.show()

    #slow plot
    plt.figure()
    plt.xlabel("Image #", fontsize = 10)
    plt.ylabel("Median Pixel Value")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_slow_light_image_week2.fits", "Slow", "light", "blue")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\avg_slow_dark_image_week2.fits", "Slow", "dark", "red")
    plot_avg(r"C:\Users\brie_\OneDrive - University of Arizona\Documents\2026 Summer Research LBT\Week_2\net_slow_image_week2.fits", "Slow", "net", "black")
    plt.legend()
    plt.grid(True)
    plt.show()

## Figures Created
![Fast Scatterplot](Week_2/fast_mode_scatterplot.png)  
![Medium Scatterplot](Week_2/medium_mode_scatterplot.png)  
![Slow Scatterplot](Week_2/slow_mode_scatterplot.png)

### Notes
-in group meeting we talked about how median is better than mean for these cases, because it will not be widely affected by outliers but here the median and mean are approximately similar  
-it takes lots of memory to open multiple (here ~10) large fits files at once and my computer was able to handle it, but good to keep in mind to close unecessary applications while running the code  
-the scatterplots I got make sense, but the adjusted difference fit still doesn't appear to go through 0 is this what we should get? Will we artificially fit the linear line up slightly to account for this?